# 🔎 Bronze Schema Profiling

Before designing the Silver layer, the raw GH Archive dataset is
profiled to understand its structure.

GH Archive contains multiple GitHub event types, and different event
types contain different JSON payloads.

The profiling process will:

1. Discover all event types.
2. Count how many events exist for each type.
3. Discover the combined JSON schema for each event type.
4. Use those schemas to decide which fields belong in Silver.

No production data is modified during this process.

In [0]:
from pyspark.sql.functions import (
    col,
    get_json_object,
    count
)


bronze_df = spark.table(
    "github_lakehouse.bronze.github_events_raw"
)

In [0]:
event_types_df = (
    bronze_df
    .select(
        get_json_object(
            col("raw_json"),
            "$.type"
        ).alias("event_type")
    )
    .groupBy("event_type")
    .agg(
        count("*").alias("event_count")
    )
    .orderBy(
        col("event_count").desc()
    )
)


display(event_types_df)

event_type,event_count
PushEvent,26537024
CreateEvent,3778827
PullRequestEvent,2173964
WatchEvent,1380565
IssueCommentEvent,1357327
DeleteEvent,874499
PullRequestReviewEvent,717285
IssuesEvent,548959
PullRequestReviewCommentEvent,422917
ForkEvent,326419


## 🧪 PushEvent Schema Discovery

`schema_of_json_agg()` scans the JSON records and creates a combined
schema.

This is useful because no individual GitHub event necessarily contains
every possible field.

In [0]:
%sql
SELECT
    schema_of_json_agg(raw_json) AS push_event_schema

FROM github_lakehouse.bronze.github_events_raw

WHERE get_json_object(raw_json, '$.type') = 'PushEvent';

push_event_schema
"STRUCT, created_at: STRING, id: STRING, org: STRUCT, payload: STRUCT, distinct: BOOLEAN, message: STRING, sha: STRING, url: STRING>>, distinct_size: BIGINT, head: STRING, push_id: BIGINT, ref: STRING, repository_id: BIGINT, size: BIGINT>, public: BOOLEAN, repo: STRUCT, type: STRING>"


In [0]:
%sql
SELECT
    get_json_object(
        raw_json,
        '$.type'
    ) AS event_type,

    schema_of_json_agg(
        raw_json
    ) AS discovered_schema

FROM github_lakehouse.bronze.github_events_raw

GROUP BY
    get_json_object(
        raw_json,
        '$.type'
    );

event_type discovered_schema ForkEvent STRUCT , created_at: STRING, id: STRING, org: STRUCT , payload: STRUCT<forkee: STRUCT<allow_forking: BOOLEAN, archive_url: STRING, archived: BOOLEAN, assignees_url: STRING, blobs_url: STRING, branches_url: STRING, clone_url: STRING, collaborators_url: STRING, comments_url: STRING, commits_url: STRING, compare_url: STRING, contents_url: STRING, contributors_url: STRING, created_at: STRING, default_branch: STRING, deployments_url: STRING, description: STRING, disabled: BOOLEAN, downloads_url: STRING, events_url: STRING, fork: BOOLEAN, forks: BIGINT, forks_count: BIGINT, forks_url: STRING, full_name: STRING, git_commits_url: STRING, git_refs_url: STRING, git_tags_url: STRING, git_url: STRING, has_discussions: BOOLEAN, has_downloads: BOOLEAN, has_issues: BOOLEAN, has_pages: BOOLEAN, has_projects: BOOLEAN, has_wiki: BOOLEAN, homepage: STRING, hooks_url: STRING, html_url: STRING, id: BIGINT, is_template: BOOLEAN, issue_comment_url: STRING, issue_events_url: STRING, issues_url: STRING, keys_url: STRING, labels_url: STRING, language: STRING, languages_url: STRING, license: STRUCT , merges_url: STRING, milestones_url: STRING, mirror_url: STRING, name: STRING, node_id: STRING, notifications_url: STRING, open_issues: BIGINT, open_issues_count: BIGINT, owner: STRUCT , private: BOOLEAN, public: BOOLEAN, pulls_url: STRING, pushed_at: STRING, releases_url: STRING, size: BIGINT, ssh_url: STRING, stargazers_count: BIGINT, stargazers_url: STRING, statuses_url: STRING, subscribers_url: STRING, subscription_url: STRING, svn_url: STRING, tags_url: STRING, teams_url: STRING, topics: ARRAY , trees_url: STRING, updated_at: STRING, url: STRING, visibility: STRING, watchers: BIGINT, watchers_count: BIGINT, web_commit_signoff_required: BOOLEAN>>, public: BOOLEAN, repo: STRUCT , type: STRING> IssueCommentEvent STRUCT , created_at: STRING, id: STRING, org: STRUCT , payload: STRUCT , external_url: STRING, html_url: STRING, id: BIGINT, name: STRING, node_id: STRING, owner: STRUCT , permissions: STRUCT<actions: STRING, actions_variables: STRING, administration: STRING, blocking: STRING, checks: STRING, codespaces: STRING, codespaces_lifecycle_admin: STRING, codespaces_metadata: STRING, codespaces_secrets: STRING, codespaces_user_secrets: STRING, contents: STRING, dependabot_secrets: STRING, deployments: STRING, discussions: STRING, emails: STRING, environments: STRING, followers: STRING, gists: STRING, git_signing_ssh_public_keys: STRING, gpg_keys: STRING, interaction_limits: STRING, issues: STRING, keys: STRING, members: STRING, merge_queues: STRING, metadata: STRING, organization_actions_variables: STRING, organization_administration: STRING, organization_announcement_banners: STRING, organization_codespaces: STRING, organization_codespaces_secrets: STRING, organization_codespaces_settings: STRING, organization_copilot_seat_management: STRING, organization_custom_org_roles: STRING, organization_custom_properties: STRING, organization_custom_roles: STRING, organization_dependabot_secrets: STRING, organization_events: STRING, organization_hooks: STRING, organization_packages: STRING, organization_personal_access_token_requests: STRING, organization_personal_access_tokens: STRING, organization_plan: STRING, organization_projects: STRING, organization_secrets: STRING, organization_self_hosted_runners: STRING, organization_user_blocking: STRING, packages: STRING, pages: STRING, plan: STRING, profile: STRING, pull_requests: STRING, repository_advisories: STRING, repository_announcement_banners: STRING, repository_custom_properties: STRING, repository_hooks: STRING, repository_projects: STRING, secret_scanning_alerts: STRING, secrets: STRING, security_events: STRING, single_file: STRING, starring: STRING, statuses: STRING, team_discussions: STRING, vulnerability_alerts: STRING, watching: STRING, workflows: STRING>, slug: STRING, updated_at: STRING>, reactions: STRUCT<`+1`: BIGINT, `-1`: BIGINT, confused: BIGINT, eyes: BIGINT